# EXP-033 · One Run all on Kaggle

Attach the existing private P2 and runtime dataset. The notebook discovers recursively extracted folders or the original ZIPs. Select a T4 GPU and enable Internet. Set **AUTHORIZE_RUN = True** and use **Run all**. The worker checks the pretrained model, validates the local resume canary and automatically runs all five folds if those checks pass.

Sealed checkpoints and logs appear in Kaggle Working during execution. A verified archive and SHA-256 receipt are exported at the end, including after a handled failure. Save the output before ending the session. Optional resume uses a previously exported and saved archive.


In [ ]:
AUTHORIZE_RUN = False

# Leave empty to find the attached exp033_inputs and exp033_runtime_oneclick folders or ZIPs.
INPUT_PATH = ""
RUNTIME_PATH = ""
RESUME_ZIP = ""  # optional, path to a previously saved run archive under /kaggle/input
RESUME_SHA256 = ""
MODEL_DIRECTORY = ""  # optional verified weights; otherwise download the pinned model
MAX_SECONDS = 32_400  # worker budget, leaving extra session time for setup and export

SOURCE_COMMIT = "9fdc3eeaf989d394ebb960a441d1d3496ac8f2ae"
RUNTIME_SHA = "055244538f4c1377f6ab9bd3dc0469018016ebf194c77d6894e074d723a11bbb"
INPUT_SHA = "3bfb284878c886c7313875f909171d3d015ee7860790375a7f873e904b106d84"
MANIFEST_SHA = "21f47c3d3c44530b2ea55d12368b46bf1050d665de607094bce591283baf1a82"


In [ ]:
import hashlib
import json
import os
from datetime import datetime, timezone
from pathlib import Path
import subprocess
import sys
import zipfile


def sha(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


KAGGLE_INPUT = Path("/kaggle/input")
RUNTIME_FILES = {
    "configs/e50_inference.json": "1c0995e0bd7d8a6d5578f5eaa1b8843ae7889301a53f9e7fa3a2081910056d92",
    "configs/e50l_inference.json": "7cfeabc32913ca786c8b4c1d7b78c396f69c205a36127f32f8c30a32176f32d7",
    "configs/exp002.json": "3a76178552857f7e178623c2390567a11f28a4625cf0d623e24e2dfa6fd150e7",
    "configs/exp003.json": "f23588f5bd91b23b15eada18600e4d3041a02565df21b4ac69f3bd5224b1fa98",
    "configs/exp004.json": "35f4ef370b5cc9d1023cb4b25ff837e74ea524f0ecdad22e1bd7658f5c35016e",
    "configs/exp005.json": "d58a6e31f963e4628849a2905ad96c2577d3e507a08d3f41fa33b184fe757452",
    "configs/exp006.json": "e1cbd4b3b760eeca81c3bb6213abf10c33f6953a9450dc8acd67b2fd2ddc5862",
    "configs/exp007.json": "1628ff4a730347867eddb9a2e15eb62cbea069b6642c90e5e93322fd18f06298",
    "configs/exp008.json": "d8a1081c6c6b6a8a3566a3af01d5b9bb899eb5c07994d0ffc25be18410524b29",
    "configs/exp010.json": "134ddf83bd49e4b6cdc4197e2d6f3e411d3ee39166e92a6d3e6e43ca31afbd27",
    "configs/exp033.json": "86a8117651fe336c436f49fcc7fca959244c877a50743d87da8bb1ffa0d07167",
    "configs/st3.json": "37089414e7b853b16f3b53f3f3c36ba0e0e611f02cf372eebf30b4c5dde0ca6b",
    "requirements-exp033.lock.txt": "52f994408dcf78de2a2e21884e539acf319fc0346a29f91d176798a4525ebfe6",
    "scripts/exp033_artifacts.py": "fa2cf980338ec7e8c961debf638e50e0c02f9c083ee71ecd18e2faeaad069559",
    "scripts/pack_exp033.py": "6048e4e2794977c39dca9b2f4e6c3ff91948ef02ab87693e0d5865d93df2e4fd",
    "scripts/run_exp033.py": "0132bb897d30fbcaceac0501ae7709df51a91c894865b1926fccb8fe4fe04944",
    "scripts/run_st3.py": "ea3abc6492bc3064b5e6b1318247345a62f5131bd24a19c0db14b37ec34855ae",
    "scripts/setup.py": "a983a3b04e6c7252a9fd495705e0b5662e3366c4c23f3cd40d5f17b5bc5a52f2",
    "scripts/setup_siglip2.py": "95aa4c7b42365587480af1c79055a9d58b567d30cae49468211407ec56dc034d",
    "src/st3/__init__.py": "70892180160e7a634b955a94d1a7175ce443d86e5180d2473859d4c706a10949",
    "src/st3/ablation.py": "0dcef19db4eb0e6e868f394a550dc52a8f5c444fd9d0fb3eb483b57acecfcd75",
    "src/st3/baseline.py": "66316bf3f5014b96d78cecd687caf67cd552968e1b7ef8b057f082258b11bc79",
    "src/st3/cli.py": "f0f2f389eee2eb8119bc1a179c80abae5b929c39fa758c3317c2ef5cc556dc7a",
    "src/st3/clip_large.py": "51b24b761254ba8db79ad32a0111f2907350ec1dba4a41a32ef6e478f22ccab3",
    "src/st3/clip_large_study.py": "bca1c8696f146b3e47c17da7aa5fc36d1c0e42651eee75b03e1b46362e20589d",
    "src/st3/contrast.py": "a1f81c55020e5de54afb4da2e41e274a0e15f7472dcd9b122575bfffbcf4a5e5",
    "src/st3/contrast_study.py": "0021c8ccc804fdacb86b044eb8bad663e9e8784ae069700d9a5b808ff5f4bcd9",
    "src/st3/coverage.py": "3736781fcf8b3e9a49ffb73b70bbe5ef0214c1add0e610e3efeeb194c911491b",
    "src/st3/coverage_study.py": "d50e70ca96a13af5c3651c5530767f2a2df1f88322850da9b53f92206b7597c2",
    "src/st3/data.py": "605556b5047c2bdc80b2d9e619635f5f7d5d3d9719738042261b6301855a0246",
    "src/st3/dino.py": "48d1dec64c8e304772d2e1bf1254f383269f29310c69144c45b9c65544ff9432",
    "src/st3/e50_encoders.py": "26dee34e53b3c7f9d4abcb3b4c2e0ab2205b54a0c0b782d48227719eca1d4f78",
    "src/st3/e50_heads.py": "54bfab819b9b3b2e0f66cd8b51ff38b09cb395c3217dd2904976852ed0aba77c",
    "src/st3/e50_inference.py": "601413de7604a99e4a69643e07496c46209a46b9ffc353eb9ca2ea1ae1e5fd98",
    "src/st3/e50l_heads.py": "efac38a943b39103186fc10aa45de8f4c3fb59bc5ea3eec4b07a58d56085f4a7",
    "src/st3/exp033.py": "e8372745ff8a1a2caeb43fc1cc2182226a1d3b2bfef53f49b0a061d40552190f",
    "src/st3/exp033_io.py": "90b8e312c5d0abd782667e46d01f22d200fa66a4fb2c07b359831bbe7058be1b",
    "src/st3/exp033_metrics.py": "6773efbfc6a76996711b58ec2838ff6503424e022bba837b6c903077454c83c5",
    "src/st3/exp033_train.py": "cc3f0e3f2822e2cc7fdbc97137ead39af58767223347dd439872a9741ebfbb25",
    "src/st3/exp033_vision.py": "cd7a03cb1c30ac5eef3ad31b7e5b2edbf4db8f3225315b1d3df142af7eb0f6cb",
    "src/st3/frozen_features.py": "ad6caaaf34636152ae84b68a3a456dbf1f5e5f9d2bcc5c1e1f03d2323a94bc60",
    "src/st3/fusion.py": "f0426dd3088af1f2c0af65df1bcd34d3f80547672ae901159a727e6496e7950b",
    "src/st3/fusion_study.py": "ec822034ee0f4c981ebb9c69726a3a58383c35a5821df0e968bc9591e108fd9e",
    "src/st3/incremental.py": "b9d9d7f4a3e5bd5218b627c39d818d4ff39400dee60ab9ae64c9edf33e3c5044",
    "src/st3/low_rank.py": "54966200e246f75e054f25127c4dc4cb9fda86a31848487747871043df0e2428",
    "src/st3/low_rank_study.py": "e4a5c8c79581791f62af1dc54f01b8f9d695d774fe4b042599e15623da8f4c89",
    "src/st3/metrics.py": "11482fb9421d80a38c88afca5c7aa9967983f5d75b6085d2e5f1f3e81779fe07",
    "src/st3/siglip2.py": "e79616a7b49d0fdd6e40119a144564143acef9781293893a9824ad16a7914c15",
    "src/st3/study_plots.py": "c2b3b4f5cd276a9037e89054ab09f24b7e5dd665793b6ce4da1eafd0f8b5f714",
    "src/st3/submission.py": "bd840e9b2deee6d490977ec37e9decb0d40dc2161a148be9dbe126bf56dabefc",
    "src/st3/visual_study.py": "666cb5de09204285da0bc06a07f3080e1860c47628ce3130498ee9a7834f80d7"
}

def mounted(value, *, allow_directory=False):
    path = Path(value).resolve()
    if (not value or not path.is_relative_to(KAGGLE_INPUT.resolve())
            or not (path.is_file() or (allow_directory and path.is_dir()))):
        raise ValueError(f"Locate the attached private input under {KAGGLE_INPUT}: {value or '(unset)'}")
    return path


def attached(value, directory_name):
    if value:
        return mounted(value, allow_directory=True)
    directories = sorted(p for p in KAGGLE_INPUT.rglob(directory_name) if p.is_dir())
    matches = directories or sorted(p for p in KAGGLE_INPUT.rglob(directory_name + ".zip") if p.is_file())
    if len(matches) != 1:
        raise ValueError(f"Expected one attached {directory_name} directory or ZIP. Set its path explicitly.")
    return mounted(str(matches[0]), allow_directory=True)


def materialize_runtime(runtime, repo):
    def write_verified(read):
        for name, expected in RUNTIME_FILES.items():
            data = read(name)
            if hashlib.sha256(data).hexdigest() != expected:
                raise ValueError(f"Pinned runtime file hash mismatch: {name}")
            dest = repo / name
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_bytes(data)

    if runtime.is_dir():
        def read(name):
            source = runtime / name
            if not source.is_file() or not source.resolve().is_relative_to(runtime.resolve()):
                raise ValueError(f"Missing or invalid pinned runtime file: {name}")
            return source.read_bytes()
        write_verified(read)
    else:
        if sha(runtime) != RUNTIME_SHA:
            raise ValueError("Runtime ZIP SHA-256 mismatch")
        with zipfile.ZipFile(runtime) as bundle:
            write_verified(bundle.read)


def input_arguments(asset, local):
    if asset.is_dir():
        # The pinned CLI requires --input-zip syntactically. Its restore_inputs()
        # takes the existing-directory branch and verifies its manifest and files.
        return ["--input-zip", local / "unused_input.zip", "--inputs", asset]
    if sha(asset) != INPUT_SHA:
        raise ValueError("P2 input ZIP SHA-256 mismatch")
    return ["--input-zip", asset, "--inputs",
            Path("/kaggle/temp") / f"exp033_inputs_{MANIFEST_SHA[:16]}"]


def run():
    if not AUTHORIZE_RUN:
        print("Review the configuration and set AUTHORIZE_RUN = True, then click Run all.")
        return
    if not 0 < MAX_SECONDS <= 36_000:
        raise ValueError("Set a finite worker budget and leave time to export.")
    runtime = attached(RUNTIME_PATH, "exp033_runtime_oneclick")
    inputs = attached(INPUT_PATH, "exp033_inputs")
    attempt = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    local = Path("/kaggle/temp") / f"exp033_{attempt}"
    repo = local / "source"
    output = Path("/kaggle/working") / f"exp033_{attempt}_checkpoints"
    repo.mkdir(parents=True)
    materialize_runtime(runtime, repo)
    input_args = input_arguments(inputs, local)
    helper = repo / "scripts/exp033_artifacts.py"
    if RESUME_ZIP:
        subprocess.run([sys.executable, str(helper), "restore", str(mounted(RESUME_ZIP)),
                        str(output), "--sha", RESUME_SHA256], check=True)
    output.mkdir(exist_ok=True)
    logs = output / f"session_{attempt}.log"
    env = os.environ.copy()
    env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    env["HF_HOME"] = str(Path("/kaggle/temp") / "exp033_model_cache")
    environment = local / "venv"
    exe = environment / "bin/python"
    artifact = Path("/kaggle/working") / f"exp033_{attempt}.zip"

    def execute(cmd, log):
        process = subprocess.Popen([str(p) for p in cmd], cwd=repo, env=env,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        try:
            for line in process.stdout:
                log.write(line)
                log.flush()
                print(line, end="")
            if process.wait() != 0:
                raise RuntimeError(f"Worker exited {process.returncode}, see {logs.name}")
        finally:
            if process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=30)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()

    try:
        with logs.open("w", encoding="utf-8") as log:
            print(f"EXP-033 worker {SOURCE_COMMIT}", file=log)
            gpu = subprocess.run(
                ["nvidia-smi", "-i", "0", "--query-gpu=name,memory.total,memory.free,driver_version", "--format=csv"],
                capture_output=True, text=True, check=True,
            )
            print(gpu.stdout, file=log, flush=True)
            print(gpu.stdout)
            if "T4" not in gpu.stdout:
                raise ValueError("Select a Kaggle T4 accelerator.")
            bootstrap = local / "uv_pkg"
            execute([sys.executable, "-m", "pip", "install", "--target", bootstrap, "uv==0.8.15"], log)
            env["PYTHONPATH"] = str(bootstrap)
            env["UV_PYTHON_INSTALL_DIR"] = str(Path("/kaggle/temp") / "exp033_python")
            uv = [sys.executable, "-m", "uv"]
            execute(uv + ["python", "install", "3.12.10"], log)
            execute(uv + ["venv", "--python", "3.12.10", environment], log)
            execute(uv + ["pip", "install", "--python", exe, "torch==2.7.1+cu126",
                          "--index-url", "https://download.pytorch.org/whl/cu126"], log)
            execute(uv + ["pip", "install", "--python", exe, "-r", repo / "requirements-exp033.lock.txt"], log)
            execute(uv + ["pip", "check", "--python", exe], log)
            execute(uv + ["pip", "freeze", "--python", exe], log)
            env.pop("PYTHONPATH")
            cmd = [exe, "-u", repo / "scripts/run_exp033.py", "run", "--authorize",
                   *input_args,
                   "--output", output, "--zip-sha", INPUT_SHA, "--manifest-sha", MANIFEST_SHA,
                   "--cache-dir", Path("/kaggle/temp") / "exp033_frozen", "--max-seconds", MAX_SECONDS]
            if MODEL_DIRECTORY:
                cmd += ["--model-dir", Path(MODEL_DIRECTORY)]
            execute(cmd, log)
            info = json.loads((output / "decision.json").read_text(encoding="utf-8"))
            print("Decision:", info["status"])
            for scope, arms in info.get("metrics", {}).items():
                print(scope, {name: (row["pooled"]["cultural_contrast"], row["pooled"]["log_loss"])
                              for name, row in arms.items()})
    except (Exception, KeyboardInterrupt) as error:
        with logs.open("a", encoding="utf-8") as log:
            print(f"Notebook stopped: {error}", file=log)
        raise
    finally:
        subprocess.run([sys.executable, str(helper), "export", str(output), str(artifact)], check=True)
        print("Export verified:", artifact.name)
        print("Archive SHA-256:", sha(artifact))
        print("Publish or download this archive before stopping the runtime. Local session storage is temporary.")


run()


### Optional resume

Attach a previously exported archive as private input and set its path and SHA-256 in the configuration. The same Run all rechecks T4 feasibility, verifies the original worker and inputs, and resumes from sealed epochs. Fold results already sealed are reused. If Kaggle destroys the session before its output is saved, checkpoints that existed only in that session are unavailable.
